#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Named-Entity Recognition

# Modelo Pre-Entrenado

In [1]:
#Importar Librerías
import spacy
from pprint import pprint

In [2]:
#Cargar Texto
text = "Orlando Bloom sorprendió con radical cambio de look: estaría en campaña para agrandar su familia con Katy Perry"

In [3]:
#Descargar Pipeline (Spanish)
!python -m spacy download es_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 74.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
#Cargar Pipeline
nlp = spacy.load("es_core_news_sm")

In [5]:
#Crear Doc Object
doc = nlp(text)

In [6]:
#Identificación de Entidades
print("Texto Original:", doc)
for ent in doc.ents:
  print(ent.text, "---", ent.label_)

Texto Original: Orlando Bloom sorprendió con radical cambio de look: estaría en campaña para agrandar su familia con Katy Perry
Orlando Bloom --- PER
Katy Perry --- PER


In [8]:
#Visualización Entidades
from spacy import displacy
displacy.render(doc, style='ent', jupyter=True)

In [9]:
#Función para revisar Entidades
def get_entities(text, pipeline):
  doc = pipeline(text)
  for ent in doc.ents:
    print(ent.text, "---", ent.label_)

In [10]:
#Ejemplo "Chileno"
text = "Orlando Bloom sorprendió con radical cambio de look: estaría en campaña para agrandar su familia con Katy Perry en Melipilla"
get_entities(text, nlp)

Orlando Bloom --- PER
Katy Perry --- PER
Melipilla --- LOC


In [11]:
#Ejemplo "Chileno"
text = "NotCo fue fundada en Melipilla"
get_entities(text, nlp)

NotCo --- MISC
Melipilla --- LOC


# Evaluación Modelo NER

In [12]:
#Librerías Evaluación
from spacy.training.example import Example
from spacy.scorer import Scorer

In [13]:
#Datos Etiquetados (Train & Test Data)
train_data = [
              ("Mi mamá odia NotCo", {"entities": [(13, 18, "ORG")]}),
              ("Katy Perry ama NotCo.", {"entities": [(0, 10, "PER"), (15, 20, "ORG")]}),
              ("Orlando Bloom compró acciones en NotCo", {"entities": [(0, 13, "PER"), (33,38, "ORG")]}),
              ("Mi hermana nació en Melipilla", {"entities": [(20,29, "GPE")]}),
              ("Katy Perry visitó Melipilla", {"entities": [(0, 10, "PER"), (18,27, "GPE")]}),
              ("Los fundadores de NotCo nacieron en Melipilla", {"entities": [(18, 23, "ORG"), (36,45, "GPE")]}),
              ("No me gustó la comida en Melipilla", {"entities": [(25,34, "GPE")]})
              ]

test_data = [
              ("Orlando Bloom compró una casa en Melipilla", {"entities": [(0, 13, "PER"), (33,42, "GPE")]}),
              ("Mi postulación a NotCo fue rechazada", {"entities": [(17,22, "ORG")]}),
              ("Mi papá fue al colegio en Melipilla", {"entities": [(26,35, "GPE")]}),
              ("El que fue a Melipilla perdió su silla", {"entities": [(13,22, "GPE")]}),
              ("No me gustan los productos de NotCo", {"entities": [(30,35, "ORG")]}),
              ("Anoche hubieron protestas contra NotCo", {"entities": [(33,38, "ORG")]}),
              ]

In [14]:
#Predecir Entidades en Test Set
evaluation_data = list()
for text, annotations in test_data:
  doc_ground_text = nlp.make_doc(text)
  example = Example.from_dict(doc_ground_text, annotations)
  example.predicted = nlp(str(example.predicted))
  evaluation_data.append(example)

In [15]:
#Evaluar performance en Test Set
scorer = Scorer()
evaluation_metrics = scorer.score(evaluation_data)
pprint(evaluation_metrics)

{'cats_auc_per_type': {},
 'cats_f_per_type': {},
 'cats_macro_auc': 0.0,
 'cats_macro_f': 0.0,
 'cats_macro_p': 0.0,
 'cats_macro_r': 0.0,
 'cats_micro_f': 0.0,
 'cats_micro_p': 0.0,
 'cats_micro_r': 0.0,
 'cats_score': 0.0,
 'cats_score_desc': 'macro F',
 'dep_las': None,
 'dep_las_per_type': None,
 'dep_uas': None,
 'ents_f': 0.25,
 'ents_p': 0.2222222222222222,
 'ents_per_type': {'GPE': {'f': 0.0, 'p': 0.0, 'r': 0.0},
                   'LOC': {'f': 0.0, 'p': 0.0, 'r': 0.0},
                   'MISC': {'f': 0.0, 'p': 0.0, 'r': 0.0},
                   'ORG': {'f': 0.5, 'p': 1.0, 'r': 0.3333333333333333},
                   'PER': {'f': 0.6666666666666666, 'p': 0.5, 'r': 1.0}},
 'ents_r': 0.2857142857142857,
 'morph_acc': None,
 'morph_micro_f': None,
 'morph_micro_p': None,
 'morph_micro_r': None,
 'morph_per_feat': None,
 'pos_acc': None,
 'sents_f': None,
 'sents_p': None,
 'sents_r': None,
 'tag_acc': None,
 'token_acc': 1.0,
 'token_f': 1.0,
 'token_p': 1.0,
 'token_r': 1.0}


# Modelo Customisado

In [16]:
#Importar Librerías
import random
from spacy.util import minibatch, compounding

In [17]:
#Revisar componentes del Pipeline
nlp.pipe_names

['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [18]:
#Deshabilitar componentes que NO vamos a customizar
pipe_exceptions = ["ner"]
unafeccted_pipes = [pipe for pipe in nlp.pipe_names if pipe not in pipe_exceptions]

In [20]:
#Customizar NER Model
with nlp.disable_pipes(*unafeccted_pipes):

  #Ciclos de Entrenamiento
  for iteration in range(30):

    #Reordenar los datos para cada iteración
    random.shuffle(train_data)
    losses = {}

    #Crear mini-batches
    batches = minibatch(train_data, size=compounding(4.0, 32.0, 1.001))

    for batch in batches:
      for text, annotation in batch:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotation)
        nlp.update(
            [example],
            losses=losses, drop=0.3)
    print("Losses", losses)

Losses {'ner': np.float32(16.278133)}
Losses {'ner': np.float32(12.2044525)}
Losses {'ner': np.float32(4.791508)}
Losses {'ner': np.float32(5.3054004)}
Losses {'ner': np.float32(4.595715)}
Losses {'ner': np.float32(3.2305272)}
Losses {'ner': np.float32(0.6577191)}
Losses {'ner': np.float32(0.13566038)}
Losses {'ner': np.float32(0.0011962786)}
Losses {'ner': np.float32(0.0030286922)}
Losses {'ner': np.float32(0.00017949556)}
Losses {'ner': np.float32(5.504948e-05)}
Losses {'ner': np.float32(0.002426707)}
Losses {'ner': np.float32(7.232965e-06)}
Losses {'ner': np.float32(5.5790963e-05)}
Losses {'ner': np.float32(0.0022677418)}
Losses {'ner': np.float32(5.4744916e-05)}
Losses {'ner': np.float32(0.00012924997)}
Losses {'ner': np.float32(7.137266e-07)}
Losses {'ner': np.float32(5.1578957e-07)}
Losses {'ner': np.float32(9.2494327e-07)}
Losses {'ner': np.float32(4.2099657e-07)}
Losses {'ner': np.float32(1.2000239e-05)}
Losses {'ner': np.float32(3.5538272e-07)}
Losses {'ner': np.float32(4.0057

In [21]:
#Predecir Entidades en Test Set
evaluation_data_custom=list()
for text, annotations in test_data:

    doc_ground_text = nlp.make_doc(text)
    example = Example.from_dict(doc_ground_text, annotations)
    example.predicted = nlp(str(example.predicted))
    evaluation_data_custom.append(example)

scorer = Scorer()
evaluation_metrics_custom = scorer.score(evaluation_data_custom)
pprint(evaluation_metrics_custom)

{'cats_auc_per_type': {},
 'cats_f_per_type': {},
 'cats_macro_auc': 0.0,
 'cats_macro_f': 0.0,
 'cats_macro_p': 0.0,
 'cats_macro_r': 0.0,
 'cats_micro_f': 0.0,
 'cats_micro_p': 0.0,
 'cats_micro_r': 0.0,
 'cats_score': 0.0,
 'cats_score_desc': 'macro F',
 'dep_las': None,
 'dep_las_per_type': None,
 'dep_uas': None,
 'ents_f': 0.9333333333333333,
 'ents_p': 0.875,
 'ents_per_type': {'GPE': {'f': 1.0, 'p': 1.0, 'r': 1.0},
                   'ORG': {'f': 1.0, 'p': 1.0, 'r': 1.0},
                   'PER': {'f': 0.6666666666666666, 'p': 0.5, 'r': 1.0}},
 'ents_r': 1.0,
 'morph_acc': None,
 'morph_micro_f': None,
 'morph_micro_p': None,
 'morph_micro_r': None,
 'morph_per_feat': None,
 'pos_acc': None,
 'sents_f': None,
 'sents_p': None,
 'sents_r': None,
 'tag_acc': None,
 'token_acc': 1.0,
 'token_f': 1.0,
 'token_p': 1.0,
 'token_r': 1.0}


In [22]:
#Ejemplo Chileno
text = "Orlando Bloom compró una casa en Melipilla con la venta de sus acciones en NotCo"
get_entities(text, nlp)

Orlando Bloom --- PER
Melipilla --- GPE
NotCo --- ORG


# Componente basada en Reglas

In [23]:
#Ejemplo Chileno
text = "Katy Perry invirtió en acciones de NotCo y Camel Secure"
get_entities(text, nlp)

Katy Perry --- PER
NotCo --- ORG
Camel Secure --- PER


In [24]:
#Listar Reglas
patterns = [{"label":"ORG", "pattern":[{"LOWER":"camel"}, {"LOWER":"secure"}]}]

In [25]:
#Incorporar EntityRuler
config = {"overwrite_ents":True}
ruler = nlp.add_pipe("entity_ruler", config=config)

In [26]:
#Revisar componentes del Pipeline
nlp.pipe_names

['tok2vec',
 'morphologizer',
 'parser',
 'attribute_ruler',
 'lemmatizer',
 'ner',
 'entity_ruler']

In [27]:
#Incorporar Patrones al EntityRuler
ruler.add_patterns(patterns)

In [28]:
#Ejemplo Chileno
text = "Katy Perry invirtió en acciones de NotCo y Camel Secure"
get_entities(text, nlp)

Katy Perry --- PER
NotCo --- ORG
Camel Secure --- ORG


# Guardar & Cargar Pipelines

In [29]:
#Guardar Pipeline
from pathlib import Path
output_dir = Path("/content/")
nlp.to_disk(output_dir)
print("Modelo guardado en:", output_dir)

Modelo guardado en: /content


In [30]:
#Cargar Pipeline
nlp_custom = spacy.load(output_dir)

In [31]:
#Ejemplo Chileno
text = "Katy Perry invirtió en acciones de NotCo y Camel Secure"
get_entities(text, nlp_custom)

Katy Perry --- PER
NotCo --- ORG
Camel Secure --- ORG
